# 03 — Rotors and Sandwich Conjugation

This notebook introduces **rotors** — the Geometric Algebra representation of rotations. We'll see how rotors avoid the gimbal lock and coordinate-awkwardness of quaternions, and how the **sandwich product** applies rotations naturally.

## Learning Objectives

- Understand rotors as even-grade elements
- Construct rotors for 2D rotations
- Apply rotors using sandwich conjugation
- Visualize the rotation action
- Compare with classical approaches (rotation matrices)

In [ ]:
# Setup
import matplotlib.pyplot as plt
import numpy as np

from amsa import Algebra

alg = Algebra.vga2d()

## 3.1 What is a Rotor?

A **rotor** is an even-grade multivector (scalar + bivector) that represents a rotation:

$$R = \cos(\theta/2) - I \sin(\theta/2)$$

where $I$ is the unit bivector (e12 in 2D).

Key properties:
- **Even grade**: only grades 0 and 2
- **Unit rotor**: $R \tilde{R} = 1$ (normalized)
- **Double-cover**: $R$ and $-R$ represent the same rotation

This is fundamentally different from matrices or quaternions — it's coordinate-free and geometrically natural.

In [ ]:
# Create a rotor for 45-degree rotation
theta = np.pi / 4  # 45 degrees

rotor = alg.multivector({
    "e": np.cos(theta / 2),
    "e12": -np.sin(theta / 2)
})

print("Rotor:", rotor.values)
print("Layout blades:", rotor.layout.blades)
print("\nScalar part (cos):", np.cos(theta/2))
print("Bivector part (sin):", -np.sin(theta/2))

## 3.2 Normalizing a Rotor

For a rotor to represent a valid rotation, it must be normalized: $R \tilde{R} = 1$, where $\tilde{R}$ is the **reverse** (swap scalar and bivector sign).

In [ ]:
# Check normalization
reverse_rotor = rotor.reverse()
product = rotor * reverse_rotor

print("Rotor:", rotor.values)
print("Reverse:", reverse_rotor.values)
print("R * reverse:", product.values)
print("\nIs normalized:", np.isclose(product.grade(0).values[0], 1.0))

In [ ]:
# Use the built-in normalized() method
rotor_normalized = rotor.normalized()

product_norm = rotor_normalized * rotor_normalized.reverse()
print("After normalization:", product_norm.grade(0).values[0])

## 3.3 The Sandwich Product

To apply a rotor to a vector, we use the **sandwich product**:

$$v' = R v \tilde{R}$$

This is called *sandwich* because the vector is sandwiched between the rotor and its reverse.

The key insight: this naturally handles rotation without needing to extract a matrix or quaternion first!

In [ ]:
# Apply rotor to a vector using sandwich
v = alg.vector([1.0, 0.0])  # point along x-axis

v_rotated = rotor_normalized.sandwich(v)

print("Original vector:", v.values)
print("Rotated vector:", v_rotated.grade(1).values)
print("\nExpected (45° rotation):", [np.cos(np.pi/4), np.sin(np.pi/4)])

## 3.4 Visualizing the Rotation

Let's visualize several vectors being rotated by the same rotor to build intuition.

In [ ]:
# Create a fan of vectors and rotate them all
angles = np.linspace(0, 2*np.pi, 12, endpoint=False)
vectors = [alg.vector([np.cos(a), np.sin(a)]) for a in angles]

# Rotate all vectors
rotated_vectors = [rotor_normalized.sandwich(v).grade(1) for v in vectors]

fig, ax = plt.subplots(figsize=(7, 7))

# Draw original vectors (blue, dashed)
for a, v in zip(angles, vectors):
    vals = v.grade(1).values
    ax.arrow(0, 0, vals[0]*0.8, vals[1]*0.8, head_width=0.05, head_length=0.03, 
             fc='blue', ec='blue', alpha=0.4, linewidth=1.5, linestyle='--')

# Draw rotated vectors (red, solid)
for a, v_rot in zip(angles, rotated_vectors):
    vals = v_rot.values
    ax.arrow(0, 0, vals[0]*0.8, vals[1]*0.8, head_width=0.05, head_length=0.03, 
             fc='red', ec='red', alpha=0.8, linewidth=2)

# Add legend
ax.arrow(0.5, 0.9, 0.2, 0, head_width=0.03, head_length=0.02, fc='blue', ec='blue', alpha=0.5)
ax.text(0.75, 0.9, 'Original', fontsize=10, color='blue')
ax.arrow(0.5, 0.8, 0.2, 0, head_width=0.03, head_length=0.02, fc='red', ec='red', alpha=0.8)
ax.text(0.75, 0.8, f'Rotated {int(np.degrees(theta))}°', fontsize=10, color='red')

ax.set_xlim(-1.3, 1.3)
ax.set_ylim(-1.3, 1.3)
ax.set_aspect('equal')
ax.set_title('Rotor Application: Sandwich Product', fontsize=12)
ax.grid(True, alpha=0.3)
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
plt.show()

## 3.5 Comparison with Rotation Matrices

Let's verify that our rotor produces the same result as a classical rotation matrix.

In [ ]:
# Create rotation matrix
theta = np.pi / 6  # 30 degrees
c, s = np.cos(theta), np.sin(theta)
R_matrix = np.array([[c, -s], [s, c]])

# Create equivalent rotor
rotor = alg.multivector({
    "e": np.cos(theta / 2),
    "e12": -np.sin(theta / 2)
}).normalized()

# Test vector
v = alg.vector([1.0, 0.5])
v_vals = v.grade(1).values

# Apply matrix
v_matrix = R_matrix @ v_vals

# Apply rotor via sandwich
v_rotor = rotor.sandwich(v).grade(1).values

print("Rotation matrix result:", v_matrix)
print("Rotor sandwich result:", v_rotor)
print("\nResults match:", np.allclose(v_matrix, v_rotor))

## 3.6 Inverse Rotation

The inverse of a rotor is simply its reverse (for unit rotors):

$$R^{-1} = \tilde{R}$$

This makes inverting a rotation trivial — no matrix inversion or quaternion conjugate needed!

In [ ]:
# Create a rotor and its inverse
theta = np.pi / 3  # 60 degrees
rotor = alg.multivector({
    "e": np.cos(theta / 2),
    "e12": -np.sin(theta / 2)
}).normalized()

rotor_inv = rotor.reverse()

# Test: rotate, then rotate back
v = alg.vector([1.0, 0.0])
v_rotated = rotor.sandwich(v)
v_back = rotor_inv.sandwich(v_rotated)

print("Original:", v.grade(1).values)
print("After forward rotation:", v_rotated.grade(1).values)
print("After inverse (back):", v_back.grade(1).values)
print("\nRestored to original:", np.allclose(v.grade(1).values, v_back.grade(1).values))

## 3.7 Composing Rotors

Rotors compose via multiplication:

$$R_{combined} = R_2 R_1$$

This is simpler than matrix multiplication or quaternion multiplication!

In [ ]:
# Compose two rotations: 30° + 45° = 75°
theta1 = np.pi / 6   # 30°
theta2 = np.pi / 4   # 45°

R1 = alg.multivector({"e": np.cos(theta1/2), "e12": -np.sin(theta1/2)}).normalized()
R2 = alg.multivector({"e": np.cos(theta2/2), "e12": -np.sin(theta2/2)}).normalized()

# Composition: R2 * R1 applies R1 first, then R2
R_combined = R2 * R1

# Extract effective angle (for unit rotor: arccos(scalar) * 2)
scalar_part = R_combined.grade(0).values[0]
effective_angle = 2 * np.arccos(scalar_part)

print("Rotor 1 (30°):", R1.values)
print("Rotor 2 (45°):", R2.values)
print("Combined:", R_combined.values)
print(f"\nEffective angle: {np.degrees(effective_angle):.1f}°")
print(f"Expected: {np.degrees(theta1 + theta2):.1f}°")

## 3.8 Why Rotors Instead of Matrices or Quaternions?

| Feature | Rotors | Matrices | Quaternions |
|---------|--------|----------|-------------|
 Coordinate-free | ✅ | ❌ | ❌ (sort of) |
 No gimbal lock | ✅ | ❌ | ✅ |
 Single type for all dimensions | ✅ | ❌ | ❌ (special case) |
 Easy inverse | $R^{-1} = \tilde{R}$ | $M^{-1}$ | $q^{-1} = q^*$ |
 Natural composition | $R_2 R_1$ | $B A$ | $q_2 q_1$ |
 Geometric interpretation | Direct | Hidden | Hidden |

The rotor's geometric meaning is immediately clear: it's an oriented plane element that rotates!

## 3.9 Summary

We covered:

- **Rotors**: even-grade elements $R = \cos(\theta/2) - I\sin(\theta/2)$
- **Sandwich product**: $v' = R v \tilde{R}$ applies rotation
- **Normalization**: ensures $R\tilde{R} = 1$
- **Inverse**: simply the reverse $\tilde{R}$
- **Composition**: rotor multiplication
- **Equivalence**: matches rotation matrices exactly

In the next notebook, we'll extend to VGA3d and explore 3D rotations.

## Exercises

### ⭐ Easy

**3.1** Create a rotor for 60° rotation and apply it to the vector `[1, 1]`. Verify the result matches the rotation matrix.

In [ ]:
# Your turn: ⭐ Exercise 3.1
theta = np.pi / 3  # 60 degrees
# TODO: Create rotor, rotate [1,1], compare with matrix
raise NotImplementedError("Implement exercise 3.1")

### ⭐⭐ Medium

**3.2** Write code to verify that for any angle θ, rotating a vector by θ then by -θ returns the original vector. Test for 5 different angles.

In [ ]:
# Your turn: ⭐⭐ Exercise 3.2
test_angles = [np.pi/8, np.pi/4, np.pi/3, np.pi/2, np.pi]
v = alg.vector([1.0, 0.5])
# TODO: Verify R(theta) * R(-theta) = identity
raise NotImplementedError("Implement exercise 3.2")

### ⭐⭐⭐ Challenge

**3.3** Write a function `rotor_from_vector_pair(u, v)` that takes two vectors and returns the rotor that rotates `u` to align with `v`. Hint: the rotor is related to $(u + v)$ and $(u \cdot v + u \wedge v)$.

In [ ]:
# Your turn: ⭐⭐⭐ Exercise 3.3
def rotor_from_vector_pair(u, v):
    """Return rotor that rotates u to align with v."""
    # TODO: Implement using geometric product
    raise NotImplementedError("Implement rotor_from_vector_pair")

# Test: rotate e1 to point at 45 degrees
u = alg.vector([1.0, 0.0])
v = alg.vector([np.cos(np.pi/4), np.sin(np.pi/4)])
R = rotor_from_vector_pair(u, v)
result = R.sandwich(u).grade(1)
print("Original:", u.grade(1).values)
print("Rotated:", result.values)
print("Target:", v.grade(1).values)
print("Aligned:", np.allclose(result.values, v.grade(1).values))

## Attribution

This notebook draws on:

- **Geometric Algebra for Computer Graphics** — John Vince, Springer 2008
  https://link.springer.com/book/10.1007/978-1-84628-997-2
- **SIGGRAPH 2019 Course Notes: Geometric Algebra for Computer Graphics** — Charles G. Gunn
  https://arxiv.org/abs/2002.04509
- **Geometric Algebra for Computer Science** — Dorst, Lewiner, et al.
  https://geometricalgebra.org/